In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [3]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.sql.functions import col

spark = (SparkSession.builder
         .appName("TugasMandiriBigDataPertemuan12")
         .master("local[*]")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv("tugas12_ulasan.csv"))

print("Jumlah data:", df.count())
df.printSchema()
df.show(5, truncate=False)

Jumlah data: 3000
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|nama_menu     |cabang    |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|Kopi Susu     |Yogyakarta|Sore       |5  |5                 |1             |
|Es Teh        |Yogyakarta|Pagi       |1  |21                |0             |
|Cappuccino    |Yogyakarta|Sore       |5  |15                |0             |
|Kentang Goreng|Semarang  |Malam      |2  |5                 |1             |
|Americano     |Yogyakarta|Malam      |2  |10                |0             |
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5

**A. Membangun ML Pipeline** *(bobot 25%)*

Buat `Pipeline` dengan tahapan: `StringIndexer` untuk `nama_menu`, `cabang`, `waktu_pesan` → `OneHotEncoder` untuk ketiganya → `VectorAssembler` (gabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit`) → `LogisticRegression` dengan `labelCol="ulasan_positif"`.

In [4]:
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx", handleInvalid="keep")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx", handleInvalid="keep")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx", handleInvalid="keep")

encoder = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_ohe", "cabang_ohe", "waktu_pesan_ohe"],
    handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["nama_menu_ohe", "cabang_ohe", "waktu_pesan_ohe", "qty", "waktu_tunggu_menit"],
    outputCol="features", handleInvalid="keep")

lr = LogisticRegression(featuresCol="features",
    labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[
    idx_menu, idx_cabang, idx_waktu, encoder, assembler, lr])

print("Pipeline Logistic Regression berhasil dibuat.")
print("Jumlah tahap:", len(pipeline_lr.getStages()))

Pipeline Logistic Regression berhasil dibuat.
Jumlah tahap: 6


**B. Pelatihan & Evaluasi** *(bobot 25%)*

Bagi data 80:20 (`seed=77`), latih pipeline, lalu evaluasi pada data testing menggunakan **accuracy**, **F1-score**, dan **AUC-ROC**. Tampilkan juga *confusion matrix* sederhana menggunakan `groupBy`.

In [6]:
train, test = df.randomSplit([0.8, 0.2], seed=77)
print("Data training:", train.count())
print("Data testing :", test.count())

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

pred_lr.select(
    "nama_menu", "cabang", "waktu_pesan", "qty",
    "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability"
).show(10, truncate=False)

Data training: 2411
Data testing : 589
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |9                 |1             |1.0       |[0.3058095939038664,0.6941904060961336] |
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253883662782276,0.3746116337217724] |
|Americano|Magelang|Malam      |2  |2                 |1             |1.0       |[0.0860151262388745,0.9139848737611255] |
|Americano|Magelang|Malam      |3  |4                 |1             |1.0       |[0.12916091627148926,0.8708390837285107]|
|Americano|Magelang|Malam      |3  |13                |0             |0.0       |[0.5224852614162954

In [7]:
print("Confusion Matrix Logistic Regression")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy(
    "ulasan_positif", "prediction"
).show()

Confusion Matrix Logistic Regression
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



**C. Model Pembanding** *(bobot 20%)*

Latih **`RandomForestClassifier`** (`numTrees=30`) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.

In [13]:
indexer = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in ["nama_menu", "cabang", "waktu_pesan"]]

encoder = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"],
    handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec",
               "qty", "waktu_tunggu_menit"],
    outputCol="features")

rf = RandomForestClassifier(
    labelCol="ulasan_positif",
    featuresCol="features",
    numTrees=30,
    seed=77)

pipeline_rf = Pipeline(stages=indexer + [encoder, assembler, rf])
model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

# Evaluasi
acc = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif", metricName="accuracy")
f1 = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif", metricName="f1")
auc = BinaryClassificationEvaluator(
    labelCol="ulasan_positif", metricName="areaUnderROC")

hasil_rf = {"Accuracy": acc.evaluate(pred_rf), "F1-score": f1.evaluate(pred_rf), "AUC-ROC": auc.evaluate(pred_rf)}

print("Random Forest")
print("Accuracy :", hasil_rf["Accuracy"])
print("F1-score :", hasil_rf["F1-score"])
print("AUC-ROC  :", hasil_rf["AUC-ROC"])

# Perbandingan model
perbandingan = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [hasil_rf["Accuracy"]],
    "F1-score": [hasil_rf["F1-score"]],
    "AUC-ROC": [hasil_rf["AUC-ROC"]]})

display(perbandingan)

Random Forest
Accuracy : 0.7962648556876061
F1-score : 0.7959027335096346
AUC-ROC  : 0.840949623624783


,Model,Accuracy,F1-score,AUC-ROC
0,Random Forest,0.796265,0.795903,0.84095


**D. Analisis Kesalahan Model** *(bobot 15%)*

Dari hasil prediksi model **terbaik** (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model **salah** memprediksi (`label != prediction`). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada `waktu_tunggu_menit` yang berada di sekitar ambang batas tertentu)?


In [16]:
salah = pred_rf.filter(col("ulasan_positif") != col("prediction"))
print("Jumlah prediksi salah:", salah.count())
salah.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction"
).show(10, truncate=False)

# Analisis waktu tunggu
salah.groupBy("waktu_tunggu_menit").count().orderBy("waktu_tunggu_menit").show()

Jumlah prediksi salah: 120
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |
|Americano|Magelang|Sore       |3  |13                |1             |0.0       |
|Americano|Magelang|Sore       |5  |6                 |0             |1.0       |
|Americano|Magelang|Sore       |5  |15                |1             |0.0       |
|Americano|Semarang|Malam      |1  |2                 |0             |1

**E. Kesimpulan Bisnis** *(bobot 15%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).

Berdasarkan hasil praktikum, Machine Learning dapat digunakan untuk membantu memprediksi ulasan pelanggan Kopi Nusantara. Model Random Forest digunakan untuk melihat kemungkinan ulasan positif berdasarkan data pesanan pelanggan. Hasil prediksi dapat menjadi bahan evaluasi untuk mengetahui faktor yang berpengaruh terhadap kepuasan pelanggan, terutama waktu tunggu pesanan. Semakin lama waktu tunggu, kemungkinan pelanggan memberikan ulasan positif cenderung menurun.

Kopi Nusantara sebaiknya meningkatkan kecepatan pelayanan, mengatur pembagian tugas karyawan, dan memperhatikan antrean pesanan saat jam ramai. Selain itu, hasil prediksi yang salah perlu diperiksa agar model dapat dievaluasi lebih lanjut. Dengan memanfaatkan data, pihak Kopi Nusantara diharapkan dapat memperbaiki kualitas pelayanan, meningkatkan kepuasan pelanggan, dan membuat keputusan yang lebih tepat.


In [17]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
